In [0]:
%sql
-- ============================================================================
-- DDL BRONZE: Creación de Esquema y Estructura de Tablas
-- ============================================================================
-- Script centralizado para la creación de infraestructura de la capa Bronze
-- Ejecutar ANTES del proceso ETL (02_Bronze_dnrpa)
-- ============================================================================

-- 1. Crear esquema Bronze si no existe
CREATE SCHEMA IF NOT EXISTS workspace.tp_dnrpa_bronze
COMMENT 'Capa Bronze - Ingesta cruda de datos desde fuentes externas (CSV, Excel)';

-- 2. Crear tabla: bronze_transferencias
-- Almacena los datos crudos de transferencias vehiculares del DNRPA
CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_bronze.bronze_transferencias (
  tramite_tipo STRING COMMENT 'Tipo de trámite (TRANSFERENCIA, INSCRIPCION, BAJA)',
  tramite_fecha STRING COMMENT 'Fecha del trámite (formato crudo)',
  fecha_inscripcion_inicial STRING COMMENT 'Fecha de inscripción inicial del vehículo',
  registro_seccional_codigo STRING COMMENT 'Código del registro seccional',
  registro_seccional_descripcion STRING COMMENT 'Descripción del registro seccional',
  registro_seccional_provincia STRING COMMENT 'Provincia del registro seccional',
  automotor_origen STRING COMMENT 'Origen del automotor (NACIONAL/IMPORTADO)',
  automotor_anio_modelo STRING COMMENT 'Año del modelo del vehículo',
  automotor_tipo_codigo STRING COMMENT 'Código de tipo de vehículo',
  automotor_tipo_descripcion STRING COMMENT 'Descripción del tipo de vehículo',
  automotor_marca_codigo STRING COMMENT 'Código de marca del vehículo',
  automotor_marca_descripcion STRING COMMENT 'Descripción de la marca',
  automotor_modelo_codigo STRING COMMENT 'Código del modelo',
  automotor_modelo_descripcion STRING COMMENT 'Descripción del modelo',
  automotor_uso_codigo STRING COMMENT 'Código de uso del vehículo',
  automotor_uso_descripcion STRING COMMENT 'Descripción del uso',
  titular_tipo_persona STRING COMMENT 'Tipo de persona titular (FISICA/JURIDICA)',
  titular_domicilio_localidad STRING COMMENT 'Localidad del domicilio del titular',
  titular_domicilio_provincia STRING COMMENT 'Provincia del domicilio del titular',
  titular_pais_nacimiento STRING COMMENT 'País de nacimiento del titular',
  titular_porcentaje_titularidad STRING COMMENT 'Porcentaje de titularidad',
  titular_domicilio_provincia_id STRING COMMENT 'ID de la provincia del domicilio',
  _rescued_data STRING COMMENT 'Datos rescatados en caso de error de parseo'
)
USING DELTA
COMMENT 'Tabla Bronze: Datos crudos de transferencias vehiculares desde archivos CSV';

-- 3. Crear tabla: tabla_historica_valores_automotor
-- Almacena los valores históricos de automotores (referencia para análisis de precios)
CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_bronze.tabla_historica_valores_automotor (
  -- Nota: Los campos se infieren del archivo Excel
  -- Esta tabla se recrea con inferSchema en el proceso ETL
  -- Aquí solo creamos la estructura básica
)
USING DELTA
COMMENT 'Tabla Bronze: Valores históricos de automotores desde archivo Excel';

-- ============================================================================
-- Resultado esperado:
-- - Schema workspace.tp_dnrpa_bronze creado
-- - Tabla bronze_transferencias creada con esquema explícito
-- - Tabla tabla_historica_valores_automotor creada (estructura básica)
-- ============================================================================

## ✅ Refactorización Completada

### Notebooks DDL Creados (Ejecutar en este orden la primera vez):

1. **DDL_Bronze** (este notebook) - `/Users/p.ciranno@gmail.com/tp-databricks-dnrpa/notebooks/01_DDL/DDL_Bronze`
   * CREATE SCHEMA workspace.tp_dnrpa_bronze
   * CREATE TABLE bronze_transferencias (esquema explícito)
   * CREATE TABLE tabla_historica_valores_automotor

2. **DDL_Silver** - `/Users/p.ciranno@gmail.com/tp-databricks-dnrpa/notebooks/01_DDL/DDL_Silver`
   * CREATE SCHEMA workspace.tp_dnrpa_silver
   * CREATE TABLE silver_transferencias (esquema explícito con tipos DATE/TIMESTAMP)

3. **DDL_Gold** - `/Users/p.ciranno@gmail.com/tp-databricks-dnrpa/notebooks/01_DDL/DDL_Gold`
   * CREATE SCHEMA workspace.tp_dnrpa_gold
   * CREATE TABLE dim_marca, dim_tipo_vehiculo, dim_modelo, dim_geografia (4 dimensiones)
   * CREATE TABLE fact_transferencias (tabla de hechos)
   * CREATE VIEW vw_transferencias_analitica (vista desnormalizada)

---

### Notebooks ETL Modificados:

1. **02_Bronze_dnrpa** - `/Users/p.ciranno@gmail.com/tp-databricks-dnrpa/notebooks/02_Bronze/02_Bronze_dnrpa`
   * ❌ Eliminado: CREATE OR REPLACE TABLE ...
   * ✅ Cambiado a: INSERT OVERWRITE TABLE (celdas 2 y 3)

2. **02_Silver_dnrpa** - `/Users/p.ciranno@gmail.com/tp-databricks-dnrpa/notebooks/03_Silver/02_Silver_dnrpa`
   * ❌ Eliminada: Celda 22 (CREATE SCHEMA)
   * ✅ Mantenido: .saveAsTable() en celda 23 (solo inserta datos)

3. **04_Gold** - `/Users/p.ciranno@gmail.com/tp-databricks-dnrpa/notebooks/04_Gold/04_Gold`
   * ❌ Eliminada: Celda 1 (CREATE SCHEMA)
   * ❌ Eliminado: CREATE OR REPLACE TABLE ... AS (celdas 2-6)
   * ✅ Cambiado a: INSERT OVERWRITE TABLE (5 tablas)

---

### Flujo de Ejecución:

**🔵 Primera vez (Setup completo):**
```
DDL_Bronze → DDL_Silver → DDL_Gold → 02_Bronze_dnrpa → 02_Silver_dnrpa → 04_Gold
```

**🟢 Ejecuciones posteriores (Solo ETL):**
```
02_Bronze_dnrpa → 02_Silver_dnrpa → 04_Gold
```

---

### Beneficios de la Refactorización:

✅ **Separación de concerns**: DDL separado de ETL  
✅ **Versionado claro**: Esquemas documentados en un solo lugar  
✅ **Ejecución más rápida**: INSERT OVERWRITE en lugar de CREATE OR REPLACE  
✅ **Documentación mejorada**: COMMENT en columnas PKs y FKs  
✅ **Vista analítica**: vw_transferencias_analitica para consultas fáciles  

---

### Ejemplo de Uso de la Vista:

```sql
SELECT 
  automotor_marca_descripcion,
  registro_seccional_provincia,
  YEAR(tramite_fecha) as anio,
  COUNT(*) as total_transferencias
FROM workspace.tp_dnrpa_gold.vw_transferencias_analitica
GROUP BY 1, 2, 3
ORDER BY 4 DESC
LIMIT 10;
```